Q learning, main purpose is to learn value of action, then using action value to determine policy.

Policy is the ultimate goal, but not the **directly** learned object.

---
### REINFORCE
> Why not make learning policy the main task?
>
> Can we use sampled rewards to update the policy in the direction of higher expected reward?

our goal:
$$
\max_\theta J(\theta)
$$

how to calculate gradient:

$$
\nabla_\theta J
=
\mathbb E\left[
R\,\nabla_\theta\log\pi_\theta(a|s)
\right]
$$

We  want **gradient ascend**, Pytorch only provides **gradient descend** so we use a **surrogate loss**:

The goal is to minizie:
> loss = -policy.log_prob(action) * reward

We know this will increase the action's probablity (covered in llm series).
> min(-policy.log_prob(action))

the larger the reward, should affect more:
> min(-policy.log_prob(action) * reward)


An example:
dance -> +1
forward -> +10

If agent picks dance, it will increase its P (according to equation above). You say:
>but dance is worse!

Yes, if we pick forward, its P will increase more!

REINFORCE doesn't need **every** sample update to be correct. It says: when do it enough time, the expected sample updates is J's gradient.

<br><br>

you can use REINFORCE with gamma dacay, but J will become:

$$
\nabla_\theta J_\gamma
=
\mathbb E\left[
\sum_t
\gamma^t G_t
\nabla_\theta\log\pi_\theta(a_t\mid s_t)
\right]
$$

This is a **different goal** than one-step J equation above. 

We didn't change the rewards themselves. We changed how rewards at different times are valued, which changes the objective we're optimizing.

For example, if we not limited to 1 step, but learn multiple-steps game. J will change to:

$$
\nabla_\theta J
=
\mathbb E_\tau\left[
\sum_t G_t\nabla_\theta\log\pi_\theta(a_t|s_t)
\right]
$$

### Issue: high variance during gradient ascend

For example, say each time check gradient is like checking compass. Say you get results: [200, 100, -100, -200], the right direction is 0.

The updates **fluctuate significantly** even though their expectation is zero. High variance makes optimization noisy and can require more samples to make reliable progress.

Lots of back and forth, wasted effort. This will make the journey very inefficient.

How to solve this?

Baseline is a nature idea:

dance -> +1
forward -> +10

why we still need to increase dance?

We can have a baseline, if you are better -> increase, worse -> decrease P.

and subtracting an **action-independent** baseline **does not change** the expected policy gradient.:

$$
\mathbb E[(R-b)\nabla\log\pi]
=
\mathbb E[R\nabla\log\pi]
$$


<br><br>

#### What should be baseline?

Say a state has 3 actions, the baseline should be the **state's expected return** (average action's return in this state).

baseline need to be **action-independent**. Compute Advantage:
$$
A_t=G_t-V(s_t)
$$
using a baseline that has **NOT been updated** with the current sample.


<br>
Note: if num of states is large, during training some state may not have data. So we can train a NN to approximate state's expected return (instead of using V-table). 


### 1999 Policy Gradient Theorem

For reward design: e.x, for action "Block", we want to know, the long term expected value:

$$
Q^\pi(s,\mathrm{Block})
=
\mathbb E[G_t\mid s_t=s,a_t=\mathrm{Block}]
$$

we can train a function (Critic) to estimate that:

$$
f_w(s,a)\approx Q^\pi(s,a)
$$

#### Q: If critic estimate good enough, will it be ok?

The real values are:
$$
Q^\pi(s,A)=1.00,\qquad Q^\pi(s,B)=1.01
$$

We estimate:

$$
f_w(s,A)=1.006,\qquad f_w(s,B)=1.004
$$

good enough?

the gradient's direction are opposite!

In [2]:
p = 0.5

def policy_gradient(q):
    return p * (1 - p) * (q[1] - q[0])

true_q = [1.00, 1.01]
estimated_q = [1.006, 1.004]

print(f"True gradient: {policy_gradient(true_q):+.4f}")
print(f"Critic gradient: {policy_gradient(estimated_q):+.4f}")


True gradient: +0.0025
Critic gradient: -0.0005
